# Step 1: Downloading Census Data
To get the data we need the census API Key the code here will ask you for the key then run and get the fields. 
to ge the Census API Key, use this website: https://api.census.gov/data/key_signup.html

In [2]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests

YEAR = 2023
STATE_FIPS = "06"  # California. Keep as a string to preserve the leading zero.
OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)

fields = {
    "B17001_001E": "poverty_universe",
    "B17001_002E": "below_poverty",
    "B01003_001E": "population",
    "B17001_001M": "poverty_universe_moe",
    "B17001_002M": "below_poverty_moe",
    "B01003_001M": "population_moe",
}

api_key = os.getenv("CENSUS_API_KEY") or getpass("Census API key: ")

response = requests.get(
    f"https://api.census.gov/data/{YEAR}/acs/acs5",
    params={
        "get": ",".join(["NAME", *fields]),
        "for": "tract:*",
        "in": f"state:{STATE_FIPS} county:*",
        "key": api_key,
    },
    timeout=90,
)

if not response.ok:
    raise RuntimeError(f"Census request failed with HTTP {response.status_code}.")

try:
    rows = response.json()
except ValueError:
    raise RuntimeError(
        "The Census API did not return data. Check that your API key is active."
    ) from None

tracts = pd.DataFrame(rows[1:], columns=rows[0]).rename(columns=fields)
tracts.head()

,NAME,poverty_universe,below_poverty,population,poverty_universe_moe,below_poverty_moe,population_moe,state,county,tract
0,Census Tract 4001; Alameda County; California,3075,134,3094,451,96,452,06,001,400100
1,Census Tract 4002; Alameda County; California,2093,164,2093,224,88,224,06,001,400200
2,Census Tract 4003; Alameda County; California,5707,310,5727,607,151,607,06,001,400300
3,Census Tract 4004; Alameda County; California,4351,343,4395,636,219,640,06,001,400400
4,Census Tract 4005; Alameda County; California,3816,397,3822,736,209,737,06,001,400500


## Step 1.2 Calculate Poverty and Check the data

In [3]:
# Census geographic identifiers are strings. A tract GEOID has 11 digits.
tracts["county_fips"] = tracts["state"] + tracts["county"]
tracts["tract_geoid"] = tracts["county_fips"] + tracts["tract"]

for col in fields.values():
    values = pd.to_numeric(tracts[col], errors="coerce")
    tracts[col] = values.where(values >= 0)  # Remove negative Census placeholders.

tracts["at_or_above_poverty"] = tracts["poverty_universe"] - tracts["below_poverty"]

valid_denominator = tracts["poverty_universe"].where(tracts["poverty_universe"] > 0)
tracts["poverty_pct"] = 100 * tracts["below_poverty"] / valid_denominator

assert tracts["tract_geoid"].str.fullmatch(r"\d{11}").all()
assert tracts["tract_geoid"].is_unique
assert tracts["poverty_pct"].dropna().between(0, 100).all()

print("Number of census tracts:", len(tracts))
print("Tracts with missing or undefined poverty rate:", tracts["poverty_pct"].isna().sum())

display(
    tracts[
        ["tract_geoid", "NAME", "population", "below_poverty",
         "at_or_above_poverty", "poverty_pct"]
    ].head()
)

tracts.to_parquet(OUT / f"tract_poverty_{STATE_FIPS}_{YEAR}.parquet", index=False)

Number of census tracts: 9129
Tracts with missing or undefined poverty rate: 76


,tract_geoid,NAME,population,below_poverty,at_or_above_poverty,poverty_pct
0,06001400100,Census Tract 4001; Alameda County; California,3094,134,2941,4.357724
1,06001400200,Census Tract 4002; Alameda County; California,2093,164,1929,7.835643
2,06001400300,Census Tract 4003; Alameda County; California,5727,310,5397,5.431926
3,06001400400,Census Tract 4004; Alameda County; California,4395,343,4008,7.883245
4,06001400500,Census Tract 4005; Alameda County; California,3822,397,3419,10.403564


# Step 2 Cleaning Pudl and Merging Pudl Data 
Pudl has given us data from EIA that is already clean but we still need to look at and merge it with the other data that we want to look at that pudl offers. 
Additionally, Pudl hasnt cleaned some of the EIA data so we might need to manually retrieve and clean that ourselves. 

In [2]:
%pip install pandas pyarrow s3fs

  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   ------------------ --------------------- 7.3/16.1 MB 35.7 MB/s eta 0:00:01
   -------------------------------------- - 15.5/16.1 MB 36.7 MB/s eta 0:00:01
   ---------------------------------------- 16.1/16.1 MB 35.7 MB/s  0:00:00
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)

  Attempting uninstall: python-dateutil

    Found existing installation: python-dateutil 4.0.0

    Uninstalling python-dateutil-4.0.0:

      Successfully uninstalled python-dateutil-4.0.0

   ----- ----------------------------------  2/15 [propcache]
   ------------- --------------------------  5/15 [fsspec]
   ------------- --------------------------  5/15 [fsspec]
   -------------------------- ------------- 10/15 

In [4]:
from pathlib import Path

import pandas as pd
import pyarrow.compute as pc
import pyarrow.dataset as ds
import s3fs

YEAR = 2023
RELEASE = "v2026.10.0"
BASE = f"pudl.catalyst.coop/{RELEASE}"

s3 = s3fs.S3FileSystem(anon=True)

def read_year(table_name, columns):
    dataset = ds.dataset(
        f"{BASE}/{table_name}.parquet",
        filesystem=s3,
        format="parquet",
    )
    return dataset.to_table(
        columns=columns,
        filter=pc.year(ds.field("report_date")) == YEAR,
    ).to_pandas()


generators = read_year(
    "out_eia__yearly_generators",
    [
        "report_date", "plant_id_eia", "generator_id",
        "operational_status_code", "energy_source_code_1",
        "fuel_type_code_pudl", "capacity_mw",
    ],
)

plants = read_year(
    "out_eia__yearly_plants",
    [
        "report_date", "plant_id_eia", "plant_name_eia",
        "sector_id_eia", "sector_name_eia",
        "state", "latitude", "longitude",
    ],
)

print(f"Generator records: {len(generators):,}")
print(f"Plant records:     {len(plants):,}")
display(generators.head())
display(plants.head())

Generator records: 33,446
Plant records:     15,112


,report_date,plant_id_eia,generator_id,operational_status_code,energy_source_code_1,fuel_type_code_pudl,capacity_mw
0,2023-01-01,1,1,SB,DFO,oil,0.9
1,2023-01-01,1,2,OP,DFO,oil,0.9
2,2023-01-01,1,3,OP,DFO,oil,0.5
3,2023-01-01,1,5,RE,DFO,oil,0.7
4,2023-01-01,1,5.1,OP,DFO,oil,0.4


,report_date,plant_id_eia,plant_name_eia,sector_id_eia,sector_name_eia,state,latitude,longitude
0,2023-01-01,67912,Claverack Creek Solar,2,IPP Non-CHP,NY,42.271000,-73.750000
1,2023-01-01,67911,Catskill Solar,2,IPP Non-CHP,NY,42.270000,-73.748000
2,2023-01-01,67910,"Timbermill Wind, LLC",2,IPP Non-CHP,NC,36.151863,-76.601040
3,2023-01-01,67909,Sherco Solar I,1,Electric Utility,MN,45.407183,-93.936502
4,2023-01-01,67908,Richmond (NY),2,IPP Non-CHP,NY,42.792865,-77.526792


## Step 2.1 Check the records before combining them
This catches a consequential error early: if a generator appears twice, its capacity could be counted twice. The sector table also lets you see exactly which commercial and industrial categories your screen will retain.

In [5]:
plant_key = ["report_date", "plant_id_eia"]
generator_key = plant_key + ["generator_id"]

assert not generators.empty and not plants.empty, "No data returned for this year."
assert generators[generator_key].notna().all().all(), "Missing generator identifier."
assert plants[plant_key].notna().all().all(), "Missing plant identifier."
assert not generators.duplicated(generator_key).any(), "Duplicate generator-year."
assert not plants.duplicated(plant_key).any(), "Duplicate plant-year."

display(
    plants[["sector_id_eia", "sector_name_eia"]]
    .drop_duplicates()
    .sort_values("sector_id_eia")
)

display(generators["operational_status_code"].value_counts(dropna=False))

,sector_id_eia,sector_name_eia
3,1,Electric Utility
0,2,IPP Non-CHP
442,2,<NA>
259,2,NAICS-22 Non-Cogen
857,3,NAICS-22 Cogen
605,3,IPP CHP
121,4,Commercial Non-CHP
737,4,Commercial NAICS Non-Cogen
1397,4,<NA>
2881,5,Commercial NAICS Cogen


operational_status_code
OP      23765
RE       4379
SB       1586
CN        755
P         487
OS        482
U         478
V         442
L         358
T         277
OA        176
TS        131
IP        116
<NA>       11
OT          3
Name: count, dtype: int64[pyarrow]

# Step 3 Select operating fossil fuel generators at potential self generation plants 
| Decision | Rule for the pilot |
|---|---|
| Potential self-generation | Keep commercial and industrial plant sectors, codes `4`, `5`, `6`, and `7`, including facilities with or without combined heat and power. |
| Operating | Use detailed status `OP`. This gives you a reproducible operating-status screen and excludes units separately classified as standby. ([PUDL 2026.2.0 documentation][1]) |
| Fossil fuel | Use the **primary fuel**. Include PUDL’s `coal` and `oil` groups and EIA primary fuel `NG` for natural gas. This implements the coal, oil, and natural-gas scope in your proposal. A unit that uses oil only as a backup fuel will not qualify on that basis. |

[1]: https://docs.catalyst.coop/pudl/en/v2026.2.0/data_dictionaries/codes_and_labels.html?utm_source=chatgpt.com "PUDL Code Metadata"

In [6]:
units = generators.merge(
    plants,
    on=plant_key,
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("Generator rows without a plant match:", units["_merge"].eq("left_only").sum())

sector = pd.to_numeric(units["sector_id_eia"], errors="coerce")
commercial_industrial = sector.isin([4, 5, 6, 7])
operating = units["operational_status_code"].eq("OP")
fossil_primary = (
    units["fuel_type_code_pudl"].isin(["coal", "oil"])
    | units["energy_source_code_1"].eq("NG")
)

eligible_mask = (
    commercial_industrial.fillna(False).astype(bool)
    & operating.fillna(False).astype(bool)
    & fossil_primary.fillna(False).astype(bool)
)

eligible = units.loc[eligible_mask].copy()

print(f"Qualifying generators: {len(eligible):,}")
print(f"Qualifying plants:     {eligible['plant_id_eia'].nunique():,}")

# Inspect the fuel codes that actually passed your screen.
display(
    eligible.groupby(["fuel_type_code_pudl", "energy_source_code_1"])
    .size()
    .rename("generators")
    .reset_index()
)

Generator rows without a plant match: 0
Qualifying generators: 1,371
Qualifying plants:     556


,fuel_type_code_pudl,energy_source_code_1,generators
0,coal,BIT,29
1,coal,PC,12
2,coal,SUB,31
3,gas,NG,1202
4,oil,DFO,93
5,oil,RFO,3
6,oil,WO,1


## 3.1 create on row per plant and save it

In [7]:
assert not eligible.empty, "No generators passed the screen. Review Cell 3 and Cell 4."

capacity = pd.to_numeric(eligible["capacity_mw"], errors="coerce")
eligible["capacity_mw"] = capacity.where(capacity >= 0)

plant_capacity = (
    eligible.groupby(plant_key, as_index=False)
    .agg(
        qualifying_generators=("generator_id", "size"),
        generators_with_mw=("capacity_mw", "count"),
        fossil_capacity_mw=("capacity_mw", "sum"),
    )
)

plant_capacity["capacity_complete"] = (
    plant_capacity["generators_with_mw"]
    == plant_capacity["qualifying_generators"]
)
plant_capacity.loc[~plant_capacity["capacity_complete"], "fossil_capacity_mw"] = pd.NA

plant_table = plants.merge(
    plant_capacity,
    on=plant_key,
    how="inner",
    validate="one_to_one",
)

plant_table["coordinates_valid"] = (
    plant_table["latitude"].between(-90, 90)
    & plant_table["longitude"].between(-180, 180)
    & ~(plant_table["latitude"].eq(0) & plant_table["longitude"].eq(0))
).fillna(False)

assert plant_table["plant_id_eia"].is_unique
assert plant_table["qualifying_generators"].sum() == len(eligible)

out = Path("data/processed")
out.mkdir(parents=True, exist_ok=True)
output_file = out / f"potential_selfgen_fossil_plants_{YEAR}.parquet"
plant_table.to_parquet(output_file, index=False)

print(f"Saved {len(plant_table):,} plants to {output_file}")
print("Plants with incomplete MW:", (~plant_table["capacity_complete"]).sum())
print("Plants missing usable coordinates:", (~plant_table["coordinates_valid"]).sum())

display(
    plant_table.loc[
        plant_table["state"].eq("CA"),
        ["plant_name_eia", "sector_name_eia", "qualifying_generators",
         "fossil_capacity_mw", "latitude", "longitude"],
    ].head(10)
)

Saved 556 plants to data\processed\potential_selfgen_fossil_plants_2023.parquet
Plants with incomplete MW: 0
Plants missing usable coordinates: 0


,plant_name_eia,sector_name_eia,qualifying_generators,fossil_capacity_mw,latitude,longitude
0,San Manuel Central Plant Cogens,Industrial NAICS Cogen,2,5.4,34.151193,-117.230250
117,Tracy 210,Industrial NAICS Non-Cogen,1,2.6,37.722010,-121.530970
120,Extreme Via Del Oro,Commercial NAICS Non-Cogen,1,1.2,37.238110,-121.785320
121,ATT Ventura Blvd,Commercial NAICS Non-Cogen,1,1.0,34.152191,-118.456078
122,ATT North Watney,Commercial NAICS Non-Cogen,1,2.0,38.236257,-122.077759
123,Extreme San Ignacio,Commercial NAICS Non-Cogen,1,1.0,37.237000,-121.785320
124,Illumina Way SD,Industrial NAICS Non-Cogen,1,3.5,32.873020,-117.199680
130,Valero Wilmington Cogeneration Plant,Industrial NAICS Cogen,1,33.5,33.778655,-118.238530
138,Berry NMW Cogens,Industrial NAICS Cogen,2,11.4,35.307555,-119.611551
145,SunSelect1,Industrial CHP,2,6.6,35.096709,-118.602184


# Step 4: Give each tract latitiude and longitude 
you can run this alone if the data folder already has stuff in it 

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd

out = Path("data/processed")
tracts = pd.read_parquet(out / "tract_poverty_06_2023.parquet")

centers_url = (
    "https://www2.census.gov/geo/docs/reference/cenpop2020/"
    "tract/CenPop2020_Mean_TR06.txt"
)

centers = pd.read_csv(
    centers_url,
    encoding="utf-8-sig",
    dtype={"STATEFP": "string", "COUNTYFP": "string", "TRACTCE": "string"},
)

# Keeping the codes as strings preserves California's leading zero.
centers["tract_geoid"] = centers["STATEFP"] + centers["COUNTYFP"] + centers["TRACTCE"]
centers = centers.rename(
    columns={"LATITUDE": "tract_lat", "LONGITUDE": "tract_lon"}
)

# A population-weighted center requires a positive population.
centers = centers.loc[centers["POPULATION"].gt(0)]

analysis = tracts.merge(
    centers[["tract_geoid", "tract_lat", "tract_lon"]],
    on="tract_geoid",
    how="left",
    validate="one_to_one",
)

has_center = (
    analysis["tract_lat"].between(-90, 90)
    & analysis["tract_lon"].between(-180, 180)
).fillna(False)

print("Total California tracts:", len(analysis))
print("Tracts with a usable population center:", has_center.sum())
print("Tracts without a usable center:", (~has_center).sum())
display(analysis[["tract_geoid", "poverty_pct", "tract_lat", "tract_lon"]].head())

Total California tracts: 9129
Tracts with a usable population center: 9097
Tracts without a usable center: 32


,tract_geoid,poverty_pct,tract_lat,tract_lon
0,06001400100,4.357724,37.859207,-122.230347
1,06001400200,7.835643,37.848290,-122.249551
2,06001400300,5.431926,37.840092,-122.254656
3,06001400400,7.883245,37.848417,-122.257294
4,06001400500,10.403564,37.848918,-122.264808


## 4.1 calculate nearby plant counts and mw at 2, 5 and 10 km
This cell uses the haversine formula to calculate approximate straight-line distances from each tract center to the plants. Keep the full plant table here so facilities just across the California border can qualify.

In [9]:
plants_geo = plant_table.loc[
    plant_table["coordinates_valid"].fillna(False),
    ["plant_id_eia", "latitude", "longitude", "fossil_capacity_mw"],
].copy()

assert plants_geo["plant_id_eia"].is_unique
assert not plants_geo.empty, "No plants have usable coordinates."
assert has_center.any(), "No tract centers matched. Check Cell 6."

print("Qualifying plants excluded for missing or invalid coordinates:",
      len(plant_table) - len(plants_geo))

plant_lat = np.radians(plants_geo["latitude"].to_numpy(dtype=float))
plant_lon = np.radians(plants_geo["longitude"].to_numpy(dtype=float))
plant_mw = plants_geo["fossil_capacity_mw"].to_numpy(dtype=float, na_value=np.nan)

radii_km = (2, 5, 10)
earth_radius_km = 6371.0
records = []

for tract in analysis.loc[has_center].itertuples(index=False):
    lat = np.radians(tract.tract_lat)
    lon = np.radians(tract.tract_lon)

    a = (
        np.sin((plant_lat - lat) / 2) ** 2
        + np.cos(lat) * np.cos(plant_lat) * np.sin((plant_lon - lon) / 2) ** 2
    )
    distance_km = 2 * earth_radius_km * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

    row = {"tract_geoid": tract.tract_geoid}
    for radius in radii_km:
        nearby = distance_km <= radius
        row[f"plant_count_{radius}km"] = int(nearby.sum())
        row[f"capacity_mw_{radius}km"] = plant_mw[nearby].sum()

    records.append(row)

proximity = pd.DataFrame(records)
tract_results = analysis.merge(proximity, on="tract_geoid", how="left",
                              validate="one_to_one")

Qualifying plants excluded for missing or invalid coordinates: 0


## check and save results

In [10]:
# Increasing the radius should only add plants.
counts = tract_results.loc[
    has_center.to_numpy(), ["plant_count_2km", "plant_count_5km", "plant_count_10km"]
]
assert (counts["plant_count_2km"] <= counts["plant_count_5km"]).all()
assert (counts["plant_count_5km"] <= counts["plant_count_10km"]).all()

display(
    tract_results[
        ["tract_geoid", "poverty_pct", "population",
         "plant_count_5km", "capacity_mw_5km"]
    ].head(10)
)

mw = tract_results["capacity_mw_5km"].dropna()
print("Tracts with known 5 km capacity:", len(mw))
print("Tracts with unknown 5 km capacity:", tract_results["capacity_mw_5km"].isna().sum())

if len(mw):
    print(f"Share with zero capacity among known tracts: {mw.eq(0).mean():.1%}")
    print("Tracts with positive capacity:", mw.gt(0).sum())

tract_results.to_parquet(out / "tract_selfgen_proximity_06_2023.parquet", index=False)

,tract_geoid,poverty_pct,population,plant_count_5km,capacity_mw_5km
0,06001400100,4.357724,3094,1.0,28.5
1,06001400200,7.835643,2093,1.0,28.5
2,06001400300,5.431926,5727,1.0,28.5
3,06001400400,7.883245,4395,1.0,28.5
4,06001400500,10.403564,3822,1.0,28.5
5,06001400600,9.459459,1957,1.0,28.5
6,06001400700,13.601272,4404,1.0,28.5
7,06001400800,13.466170,4583,1.0,28.5
8,06001400900,11.664244,2752,1.0,28.5
9,06001401000,16.917178,6529,1.0,28.5


Tracts with known 5 km capacity: 9097
Tracts with unknown 5 km capacity: 32
Share with zero capacity among known tracts: 82.5%
Tracts with positive capacity: 1593


## 4.2  Look at the California results at all three distances
It shows how often a tract has a nearby plant and how much capacity is nearby when capacity is positive.

In [11]:
summary_rows = []

for radius in (2, 5, 10):
    count = tract_results[f"plant_count_{radius}km"].dropna()
    mw = tract_results[f"capacity_mw_{radius}km"].dropna()
    positive_mw = mw[mw > 0]

    summary_rows.append({
        "radius_km": radius,
        "tracts_with_known_count": len(count),
        "tracts_with_any_plant": count.gt(0).sum(),
        "pct_with_any_plant": 100 * count.gt(0).mean(),
        "tracts_missing_mw": len(tract_results) - len(mw),
        "pct_zero_mw_among_known": 100 * mw.eq(0).mean(),
        "tracts_with_positive_mw": len(positive_mw),
        "median_positive_mw": positive_mw.median(),
        "max_mw": mw.max(),
    })

radius_summary = pd.DataFrame(summary_rows)
display(radius_summary.round(1))

,radius_km,tracts_with_known_count,tracts_with_any_plant,pct_with_any_plant,tracts_missing_mw,pct_zero_mw_among_known,tracts_with_positive_mw,median_positive_mw,max_mw
0,2,9097,382,4.2,32,95.8,382,5.3,405.0
1,5,9097,1593,17.5,32,82.5,1593,7.0,495.9
2,10,9097,3666,40.3,32,59.7,3666,15.9,585.1


# Q1
“Among residents below poverty, X% lived in a tract whose population center was within 5 km of a qualifying plant, compared with Y% of residents at or above poverty.”

In [12]:
people = tract_results.dropna(
    subset=["plant_count_5km", "below_poverty", "at_or_above_poverty"]
).copy()

near = people["plant_count_5km"].gt(0)
groups = {
    "Below poverty": "below_poverty",
    "At or above poverty": "at_or_above_poverty",
}

rows = []
for group, col in groups.items():
    total = people[col].sum()
    living_near = people.loc[near, col].sum()

    rows.append({
        "poverty_group": group,
        "residents_in_included_tracts": total,
        "residents_in_nearby_tracts": living_near,
        "pct_in_nearby_tracts": 100 * living_near / total if total > 0 else np.nan,
    })

public_health = pd.DataFrame(rows).set_index("poverty_group")
display(public_health.round(1))

pct = public_health["pct_in_nearby_tracts"]
gap = pct["Below poverty"] - pct["At or above poverty"]

print(f"Difference (below minus at/above poverty): {gap:+.1f} percentage points")
print(f"Tracts included: {len(people):,} of {len(tract_results):,}")

,residents_in_included_tracts,residents_in_nearby_tracts,pct_in_nearby_tracts
poverty_group,,,
Below poverty,4610600,772709,16.8
At or above poverty,33918852,5603116,16.5


Difference (below minus at/above poverty): +0.2 percentage points
Tracts included: 9,097 of 9,129


# Questions 
| Part | Question | Tracts included | Model |
|---|---|---|---|
| **1. Presence** | Are higher-poverty tracts more likely to have any qualifying capacity within 5 km? | All tracts with known MW and complete predictors | Logistic mixed model with a county random intercept |
| **2. Amount** | Among tracts with nearby capacity, do higher-poverty tracts have more MW nearby? | Tracts with positive MW and complete predictors | Linear mixed model of `log(capacity_mw_5km)` with a county random intercept |

if positive tracts are concentrated in only a few counties, the positive-capacity model has limited information for estimating between-county variation, even if it has many tract rows. If every tract has the same any_mw value, skip the logistic model. Missing MW is excluded from both models here. A tract with an unknown total should never silently become a zero.

In [14]:
source = tract_results

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

radius = 5
mw_col = f"capacity_mw_{radius}km"

cols = ["tract_geoid", "county_fips", "state", "poverty_pct", "population", mw_col]
d = source[cols].copy()
assert d["tract_geoid"].is_unique, "Expected one row per tract."

# Convert nullable pandas/PyArrow columns to ordinary numeric types for statsmodels.
for col in ["poverty_pct", "population", mw_col]:
    d[col] = pd.to_numeric(d[col], errors="coerce").astype("float64")

d = d.dropna(subset=cols).copy()
d = d.loc[d["population"].gt(0) & d[mw_col].ge(0)].copy()
d["county_fips"] = d["county_fips"].astype(str)
d["state"] = d["state"].astype(str)

# A one-unit increase represents 10 percentage points, e.g. 10% to 20% poverty.
d["poverty_10pp"] = (d["poverty_pct"] - d["poverty_pct"].mean()) / 10
log_pop = np.log(d["population"])
d["log_pop_c"] = log_pop - log_pop.mean()

d["any_mw"] = d[mw_col].gt(0).astype(int)
positive = d.loc[d["any_mw"].eq(1)].copy()
positive["log_mw"] = np.log(positive[mw_col])

by_county = d.groupby("county_fips")["any_mw"].agg(["size", "sum"])
by_county.columns = ["tracts", "positive_tracts"]

print(f"Tracts used: {len(d):,} of {len(source):,}")
print(f"Zero-MW tracts: {(d['any_mw'] == 0).sum():,}")
print(f"Positive-MW tracts: {len(positive):,}")
print(f"Counties represented: {len(by_county)}")
print(f"Counties with positive MW: {positive['county_fips'].nunique()}")
print("Counties with both zero and positive tracts:",
      by_county["positive_tracts"].between(1, by_county["tracts"] - 1).sum())
display(by_county.sort_values("positive_tracts", ascending=False).head(10))

Tracts used: 9,053 of 9,129
Zero-MW tracts: 7,469
Positive-MW tracts: 1,584
Counties represented: 58
Counties with positive MW: 18
Counties with both zero and positive tracts: 18


,tracts,positive_tracts
county_fips,,
06037,2471,435
06073,732,235
06075,240,205
06059,612,128
06085,408,116
06071,465,99
06001,377,62
06111,188,47
06013,240,40


# Fit the linear mixed model 
I’d run Part 2 first because it follows the statsmodels workflow you already know. Fit an ordinary regression and a county random-intercept model on exactly the same positive tracts. Use maximum likelihood (reml=False) for the mixed-model comparison.

In [15]:
predictors = "poverty_10pp + log_pop_c"

# For a small multistate study, allow each state its own baseline.
# With California alone, there is no state effect to estimate.
if positive["state"].nunique() > 1:
    predictors += " + C(state)"

amount_formula = f"log_mw ~ {predictors}"

amount_ols = smf.ols(amount_formula, data=positive).fit()

amount_mixed = smf.mixedlm(
    amount_formula,
    data=positive,
    groups=positive["county_fips"],
    re_formula="1",
).fit(reml=False, method="powell", disp=False)

print(amount_mixed.summary())
print("Converged:", amount_mixed.converged)

display(pd.DataFrame({
    "model": ["OLS", "County random intercept"],
    "tracts": [amount_ols.nobs, amount_mixed.nobs],
    "BIC": [amount_ols.bic, amount_mixed.bic],
}).round(2))

county_var = amount_mixed.cov_re.iloc[0, 0]
residual_var = amount_mixed.scale
print(f"Adjusted county ICC on the log-MW scale: "
      f"{county_var / (county_var + residual_var):.3f}")

beta = amount_mixed.params["poverty_10pp"]
lo, hi = amount_mixed.conf_int().loc["poverty_10pp"]
print(f"Difference in geometric mean MW per 10 percentage points of poverty: "
      f"{100 * np.expm1(beta):+.1f}% "
      f"(95% CI {100 * np.expm1(lo):+.1f}% to {100 * np.expm1(hi):+.1f}%)")

         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: log_mw    
No. Observations: 1584    Method:             ML        
No. Groups:       18      Scale:              1.7571    
Min. group size:  1       Log-Likelihood:     -2723.7061
Max. group size:  435     Converged:          Yes       
Mean group size:  88.0                                  
--------------------------------------------------------
                Coef. Std.Err.   z   P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept       2.090    0.240 8.703 0.000  1.619  2.561
poverty_10pp    0.096    0.035 2.749 0.006  0.028  0.165
log_pop_c       0.062    0.075 0.829 0.407 -0.085  0.210
Group Var       0.955    0.260                          

Converged: True


,model,tracts,BIC
0,OLS,1584.0,5747.07
1,County random intercept,1584.0,5484.25


Adjusted county ICC on the log-MW scale: 0.352
Difference in geometric mean MW per 10 percentage points of poverty: +10.1% (95% CI +2.8% to +17.9%)


# Fit the logistic mixed model for any nearby capacity
For Part 1, any_mw is binary. Statsmodels provides BinomialBayesMixedGLM for a logistic model with random effects. The code also fits ordinary logistic regression so you can see how the estimated poverty association changes when you allow county baselines to differ. The mixed model uses Bayesian variational estimation, so an ordinary BIC comparison with the logistic GLM would be inappropriate.

In [16]:
presence_predictors = "poverty_10pp + log_pop_c"
if d["state"].nunique() > 1:
    presence_predictors += " + C(state)"

presence_formula = f"any_mw ~ {presence_predictors}"

presence_glm = smf.glm(
    presence_formula, data=d, family=sm.families.Binomial()
).fit()

presence_model = BinomialBayesMixedGLM.from_formula(
    presence_formula,
    {"county": "0 + C(county_fips)"},
    d,
    vcp_p=0.5,
)
presence_mixed = presence_model.fit_vb()

i = presence_model.exog_names.index("poverty_10pp")
beta = presence_mixed.fe_mean[i]
sd = presence_mixed.fe_sd[i]

print("Ordinary logistic poverty odds ratio:",
      round(np.exp(presence_glm.params["poverty_10pp"]), 3))
print(f"County mixed-model poverty odds ratio: {np.exp(beta):.3f}")
print(f"Approximate 95% credible interval: "
      f"{np.exp(beta - 1.96 * sd):.3f} to {np.exp(beta + 1.96 * sd):.3f}")

# vcp_mean is on the log standard-deviation scale.
county_sd = np.exp(presence_mixed.vcp_mean[0])
icc_logistic = county_sd**2 / (county_sd**2 + np.pi**2 / 3)
print(f"Approximate county ICC on the latent logistic scale: {icc_logistic:.3f}")

Ordinary logistic poverty odds ratio: 0.993
County mixed-model poverty odds ratio: 1.079
Approximate 95% credible interval: 1.016 to 1.145
Approximate county ICC on the latent logistic scale: 0.676
